# Stage 6 — The Transformer Block

One GPT layer looks like:

```
x → LayerNorm → MHA  →  + residual
  → LayerNorm → FFN  →  + residual
```

This is the **pre-norm** variant (LN before each sublayer), used by GPT-2 onward — more stable to train than post-norm.

Pieces we build here:
1. `LayerNorm` (from scratch, to see what it does).
2. `GELU` activation.
3. `FeedForward` block (the position-wise MLP, `4x` hidden expansion).
4. `TransformerBlock` putting it all together with residuals + dropout.
5. Sanity-check the forward shape on tokenized Mahabharata.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

device: mps


### MultiHeadAttention (carried over from Stage 5)

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_len, num_heads, dropout=0.0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = d_out // num_heads
        self.d_out     = d_out
        self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_len, context_len), diagonal=1).bool()
        )

    def forward(self, x):
        B, T, _ = x.shape
        q = self.W_q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(-2, -1)
        scores = scores.masked_fill(self.mask[:T, :T], float('-inf'))
        weights = torch.softmax(scores / (self.head_dim ** 0.5), dim=-1)
        weights = self.dropout(weights)
        ctx = (weights @ v).transpose(1, 2).contiguous().view(B, T, self.d_out)
        return self.out_proj(ctx)

## 1) LayerNorm from scratch

For each token's embedding vector, subtract the mean and divide by the std (along the feature axis), then apply a learned scale `γ` and shift `β`. Unlike BatchNorm, no cross-batch statistics — so it works with any batch size and any sequence length.

In [3]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps   = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_hat + self.shift

ln = LayerNorm(5)
x = torch.randn(2, 4, 5)
y = ln(x)
print('post-LN mean (≈0):', y.mean(dim=-1).detach())
print('post-LN var  (≈1):', y.var(dim=-1, unbiased=False).detach())

post-LN mean (≈0): tensor([[ 0.0000e+00, -2.3842e-08,  2.3842e-08,  3.8743e-08],
        [ 2.3842e-08,  2.3842e-08,  5.9605e-09, -2.3842e-08]])
post-LN var  (≈1): tensor([[0.9999, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]])


## 2) GELU activation

Smooth approximation to ReLU used by GPT-2/3. We implement the **tanh** approximation (matches GPT-2 exactly):

`GELU(x) ≈ 0.5 * x * (1 + tanh(√(2/π) * (x + 0.044715 * x³)))`

In [4]:
import math

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            math.sqrt(2 / math.pi) * (x + 0.044715 * x.pow(3))
        ))

# Quick sanity check: should match torch's approximate gelu within float tolerance.
x = torch.linspace(-3, 3, 7)
ours  = GELU()(x)
ref   = F.gelu(x, approximate='tanh')
print('max abs diff:', (ours - ref).abs().max().item())

max abs diff: 0.0


## 3) Feed-forward (position-wise MLP)

Two linear layers with a `4x` hidden expansion and GELU in between. Applied independently to every position — that's the "position-wise" part.

In [5]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            GELU(),
            nn.Linear(4 * emb_dim, emb_dim),
        )

    def forward(self, x):
        return self.net(x)

ffn = FeedForward(emb_dim=8)
print(ffn(torch.randn(2, 3, 8)).shape)  # (2, 3, 8)

torch.Size([2, 3, 8])


## 4) The TransformerBlock

Pre-norm, with residual connections around both sublayers:

```python
x = x + dropout(attn(ln1(x)))
x = x + dropout(ffn(ln2(x)))
```

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = LayerNorm(cfg['emb_dim'])
        self.attn = MultiHeadAttention(
            d_in=cfg['emb_dim'], d_out=cfg['emb_dim'],
            context_len=cfg['context_len'],
            num_heads=cfg['n_heads'],
            dropout=cfg['drop_rate'],
            qkv_bias=cfg.get('qkv_bias', False),
        )
        self.ln2  = LayerNorm(cfg['emb_dim'])
        self.ffn  = FeedForward(cfg['emb_dim'])
        self.drop = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x

## 5) Sanity check on real tokens

Load the Mahabharata token IDs from Stage 4, embed them, push through one transformer block — confirm shapes line up and params look reasonable.

In [7]:
import numpy as np
from pathlib import Path

CFG = {
    'vocab_size'  : 50257,
    'context_len' : 128,
    'emb_dim'     : 256,
    'n_heads'     : 8,
    'n_layers'    : 4,
    'drop_rate'   : 0.1,
    'qkv_bias'    : False,
}

ids_path = Path('data/processed/train.bin')
if ids_path.exists():
    train_ids = np.fromfile(ids_path, dtype=np.uint16).astype(np.int64)
else:
    train_ids = np.random.randint(0, CFG['vocab_size'], size=10_000)

B, T = 4, CFG['context_len']
starts = np.random.randint(0, len(train_ids) - T - 1, size=B)
xb = torch.tensor(np.stack([train_ids[i:i+T] for i in starts]), dtype=torch.long, device=device)

tok_emb = nn.Embedding(CFG['vocab_size'], CFG['emb_dim']).to(device)
pos_emb = nn.Embedding(CFG['context_len'], CFG['emb_dim']).to(device)
block   = TransformerBlock(CFG).to(device)

pos = torch.arange(T, device=device)
x   = tok_emb(xb) + pos_emb(pos)
y   = block(x)

print('input :', xb.shape)
print('embed :', x.shape)
print('output:', y.shape)
print('block params:', sum(p.numel() for p in block.parameters()))

input : torch.Size([4, 128])
embed : torch.Size([4, 128, 256])
output: torch.Size([4, 128, 256])
block params: 788992


Next: `07_gpt_and_training.ipynb` — stack `n_layers` of these blocks into a full GPT and train it on the Mahabharata.